In [1]:
from docx import Document

In [2]:
sample_file_path = '/Users/ephraimmeiri/Downloads/R Wieder mishnah edition processing/Sample Eduyot Critical Edition.docx'
sample_docx= Document(sample_file_path)

In [3]:
def test():
    lines = []
    for para in sample_docx.paragraphs:
        if para.contains_page_break:
            break
        # print(para.text)
        if para.text == '':
            for run in para.runs:
                print("r",run)
                for item in run.iter_inner_content():
                    print(item)
        lines.append(para.text)
    a=0
    b=0
    for i in range(len(lines)):
        if 'כ"י של עדיות' in lines[i]:
            print(i,'כ"י של עדיות')
            a=i
        if 'מקבילות' in lines[i]:
            print(i,'מקבילות')
            b=i
        # check for section break 
    
    
    last = (b-a)-1
    print(lines[:a])
    print(b-a,(len(lines)-a)/2) 
    print(lines[a],lines[b])
    print(lines[a+1],lines[b+1])
    print(lines[a+last],lines[b+last])
test()

4 כ"י של עדיות
18 מקבילות
["מסכזתהז עזדיזותז פר' א'", "\tהרבה  וחכמ' אומ' לא כדברי זה ולא כדברי \tזה    אלא    מעת     לעת    ממעטת    על    יד 1\tמפקידה \t לפקידה     ומפקידה      לפקידה ", '', '']
14 16.5
	כ"י של עדיות מקבילות
29	הרבה | מ הרב'. מ הרב'.
	לפקידה | ר לפקיד' מ לפקי'. מ ומפקי' ד ומפקיד'.


In [132]:
def process_document(file_path,verbose=True):
    doc = Document(file_path)
    pges = []
    lines = []
    for para in doc.paragraphs:
        if para.contains_page_break:
            body = []
            cola = []
            colb = []
            a=0
            b=0
            for i in range(len(lines)):
                if 'כ"י של עדיות' in lines[i].text or 'כת"י של עדיות' in lines[i].text:
                    a=i
                if 'מקבילות' in lines[i].text:
                    b=i
                    num_lines = (b-a)
                    body = lines[:a]
                    cola= lines[a:b]
                    colb= lines[b:b+num_lines+1]
                    parallels = lines[b+num_lines+1:]
                    if  parallels and parallels[0] != '':
                        jump = 0
                        for k in range(len(parallels)):
                            if parallels[k].text == '':
                                jump = k
                                break
                        print("Uneven lines",len(pges),"jump",jump)
                        for j in range(jump):
                            colb.append(parallels[j])
                        parallels = parallels[jump:]
                    if verbose:
                        print(len(body),len(cola),len(colb),len(parallels))
                    pges.append([body,cola,colb,parallels])
            lines = [para]
        else:
            lines.append(para)
    return pges

In [80]:
pages_sample = process_document(sample_file_path)

Uneven lines 0 jump 0
4 14 15 4
Uneven lines 1 jump 0
3 29 30 4
Uneven lines 2 jump 0
3 32 33 2


In [83]:
def create_output_document(pges):
    output_doc = Document()
    
    for page_num, page in enumerate(pges, 1):
        output_doc.add_heading(f'Page {page_num}', level=1)
        
        # Add body text
        if page[0]:
            output_doc.add_paragraph("Body Text:")
            for para in page[0]:
                output_doc.add_paragraph(para.text)
        
        # Create table for columns
        if page[1] and page[2]:
            output_doc.add_paragraph("Columns:")
            table = output_doc.add_table(rows=len(page[1]), cols=2)
            table.style = 'Table Grid'
            for i, (cola_para, colb_para) in enumerate(zip(page[1], page[2])):
                table.cell(i, 0).text = cola_para.text
                table.cell(i, 1).text = colb_para.text
        
        # Add parallels
        if page[3]:
            output_doc.add_paragraph("Parallels:")
            for para in page[3]:
                output_doc.add_paragraph(para.text)
        
        # Add page break
        output_doc.add_page_break()
    
    return output_doc

In [84]:
sample_table = create_output_document(pages_sample)
sample_table.save('/Users/ephraimmeiri/Downloads/R Wieder mishnah edition processing/Sample Eduyot Critical Edition Output.docx')

In [102]:
from copy import deepcopy
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

def copy_run_format(new_run, old_run):
    # Copy basic formatting
    # new_run.bold = old_run.bold
    new_run.font.bold = old_run.font.bold
    new_run.font.cs_bold = old_run.font.cs_bold
    new_run.italic = old_run.italic
    new_run.underline = old_run.underline
    new_run.font.name = old_run.font.name
    new_run.font.size = old_run.font.size
    new_run.font.color.rgb = old_run.font.color.rgb
    
    # Copy superscript/subscript
    if old_run.font.superscript:
        new_run.font.superscript = True
    if old_run.font.subscript:
        new_run.font.subscript = True
    
    # Copy style if it exists
    if old_run.style:
        new_run.style = old_run.style
    
        # Explicitly set bold at the XML level
    if old_run.bold or (old_run.font.bold is not None and old_run.font.bold):
        b = OxmlElement('w:b')
        new_run._element.rPr.append(b)
        new_run.font.bold = True
    
    # Copy any other custom XML properties
    if old_run._element.rPr is not None:
        for child in old_run._element.rPr.getchildren():
            if child.tag not in [qn('w:t'), qn('w:br')]:  # Exclude text and line breaks
                new_child = deepcopy(child)
                if new_run._element.rPr is None:
                    new_run._element.get_or_add_rPr()
                new_run._element.rPr.append(new_child)
            
    # Copy other properties that might be relevant
    new_run.font.all_caps = old_run.font.all_caps
    new_run.font.double_strike = old_run.font.double_strike
    new_run.font.emboss = old_run.font.emboss
    new_run.font.imprint = old_run.font.imprint
    new_run.font.math = old_run.font.math
    new_run.font.outline = old_run.font.outline
    new_run.font.shadow = old_run.font.shadow
    new_run.font.small_caps = old_run.font.small_caps
    new_run.font.strike = old_run.font.strike

def copy_paragraph_format(new_para, old_para):
    new_para.paragraph_format.alignment = old_para.paragraph_format.alignment
    new_para.paragraph_format.line_spacing = old_para.paragraph_format.line_spacing
    new_para.paragraph_format.space_before = old_para.paragraph_format.space_before
    new_para.paragraph_format.space_after = old_para.paragraph_format.space_after
    new_para.paragraph_format.left_indent = old_para.paragraph_format.left_indent
    new_para.paragraph_format.right_indent = old_para.paragraph_format.right_indent
    new_para.style = old_para.style

def add_formatted_text_to_cell(cell, paragraph):
    new_para = cell.add_paragraph()
    copy_paragraph_format(new_para, paragraph)
    for run in paragraph.runs:
        new_run = new_para.add_run(run.text)
        copy_run_format(new_run, run)

def create_output_document(pges):
    output_doc = Document()
    
    for page_num, page in enumerate(pges, 1):
        output_doc.add_heading(f'Page {page_num}', level=1)
        
        # Add body text
        if page[0]:
            output_doc.add_paragraph("Body Text:")
            for para in page[0]:
                new_para = output_doc.add_paragraph()
                copy_paragraph_format(new_para, para)
                for run in para.runs:
                    new_run = new_para.add_run(run.text)
                    copy_run_format(new_run, run)
        
        # Create table for columns
        if page[1] and page[2]:
            output_doc.add_paragraph("Columns:")
            table = output_doc.add_table(rows=len(page[1]), cols=2)
            table.style = 'Table Grid'
            for i, (cola_para, colb_para) in enumerate(zip(page[1], page[2])):
                add_formatted_text_to_cell(table.cell(i, 0), colb_para)
                add_formatted_text_to_cell(table.cell(i, 1), cola_para)
        
        # # Add parallels
        # if page[3]:
        #     output_doc.add_paragraph("Parallels:")
        #     for para in page[3]:
        #         new_para = output_doc.add_paragraph()
        #         copy_paragraph_format(new_para, para)
        #         for run in para.runs:
        #             new_run = new_para.add_run(run.text)
        #             copy_run_format(new_run, run)
        
        # Add page break
        output_doc.add_page_break()
    
    return output_doc
sample_table2 = create_output_document(pages_sample)
sample_table2.save('/Users/ephraimmeiri/Downloads/R Wieder mishnah edition processing/Sample Eduyot Critical Edition Output 3.docx')

In [55]:
pages_sample[1][0][0].split()

['ממעטת',
 'על',
 'יד',
 'מעת',
 'לעת:',
 "ב'",
 'כל',
 'אשה',
 'שיש',
 'לה',
 'ווסת',
 'דייה',
 'שעתה',
 'הcמשמשת',
 'בעידין',
 'הרי',
 'זו',
 'כפקידה',
 'וממעטת',
 'על',
 '5',
 'יד',
 'מעת',
 'לעת',
 'ועל',
 'יד',
 'מפקידה',
 'לפקידה',
 "ג'",
 'שמי',
 "אומ'",
 'מקב',
 'חלה',
 'הלל',
 "או'",
 'מקביים']

In [118]:
full_file_path = '/Users/ephraimmeiri/Downloads/R Wieder mishnah edition processing/Eduyot Critical Edition.docx'
all_pages = process_document(full_file_path)

Uneven lines 0 jump 1
4 26 28 4
Uneven lines 1 jump 0
3 29 30 4
Uneven lines 2 jump 0
3 32 33 2
3 33 33 0
3 29 12 0
Uneven lines 5 jump 0
3 25 26 5
4 30 11 0
Uneven lines 7 jump 0
3 29 30 5
Uneven lines 8 jump 1
3 25 27 4
Uneven lines 9 jump 0
3 29 30 4
Uneven lines 10 jump 0
3 33 34 3
Uneven lines 11 jump 2
3 29 32 1
Uneven lines 12 jump 1
3 28 30 1
Uneven lines 13 jump 0
4 31 32 4
Uneven lines 14 jump 1
3 30 32 4
Uneven lines 15 jump 4
3 28 33 1
Uneven lines 16 jump 3
3 26 30 4
Uneven lines 17 jump 0
3 28 29 5
Uneven lines 18 jump 0
3 28 29 1
3 27 28 0
3 38 35 0
3 31 31 0
Uneven lines 22 jump 1
3 31 33 1
Uneven lines 23 jump 0
3 25 26 1
Uneven lines 24 jump 1
5 31 33 3
Uneven lines 25 jump 0
3 27 28 9
Uneven lines 26 jump 2
3 31 34 1
Uneven lines 27 jump 0
3 29 30 4
Uneven lines 28 jump 0
3 26 27 5
Uneven lines 29 jump 1
3 30 32 4
Uneven lines 30 jump 0
3 27 28 4
Uneven lines 31 jump 1
3 29 31 4
Uneven lines 32 jump 0
3 30 31 5
Uneven lines 33 jump 0
3 30 31 4
Uneven lines 34 jump 0


In [111]:
for page in all_pages:
    body_txt = " ".join([p.text for p in page[0]])
    non_empty_Alines = [line.text for line in page[1] if line.text != '']
    print(len(non_empty_Alines),len(body_txt.split()))

22 38
24 36
28 33
27 35
25 31
21 35
25 36
25 32
21 24
26 24
29 35
26 25
25 24
27 29
26 32
25 27
23 27
25 28
23 25
24 30
29 28
26 30
27 37
22 24
26 39
23 32
27 31
25 35
22 28
26 34
23 34
25 34
26 31
26 32
24 34
25 34
23 45
29 33
23 30
20 38
24 30
24 29
28 36
27 31
20 25
23 29
24 33
23 25
22 22
24 29
23 31
25 28
27 28
26 30
29 32
22 26
24 31
26 28
26 33
4 35
26 31
26 32
26 32
24 26
27 37
23 25
24 31
28 30
19 24
27 35
23 37
24 26
24 27
27 29
25 32
24 33
24 30
24 33
21 27
21 29
27 28
22 28
29 32
26 28
22 24
24 33
22 34
20 35
24 25
24 30
23 34
25 33
26 33
23 32
25 34
26 35
26 32
23 25
24 36
24 32
26 40
22 38
25 31
25 36
20 32
24 32
28 35
22 37
23 24
22 33
21 33
27 34
30 27
23 31
22 24
20 24
19 20
22 25
24 33
25 29
27 32
20 33
20 30
24 33
20 26
21 32
29 31
23 30
25 37
23 31
19 26


In [123]:
lst = [1,2,3,4,5,6,7,8,9,10]
print(lst[:-1])

[1, 2, 3, 4, 5, 6, 7, 8, 9]


In [157]:
import pandas as pd
pages_sample= process_document(sample_file_path)
page = pages_sample[0]
pd.DataFrame([[p.text for p in page[1]],[p.text for p in page[2]]])    

Uneven lines 0 jump 0
4 14 15 4
Uneven lines 1 jump 0
3 29 30 4
Uneven lines 2 jump 0
3 32 33 2


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,"\tכ""י של עדיות",29\tהרבה | מ הרב'.,\tוחכמ' | פ וחכמי' ר א ק ש יא מ ד וחכמים.,\tאומ' | פ מ או' ר ש יא אומרין ק אומרי' ד אומרים.,\tכדברי2 | מ כדב'.,,30\tאלא | מ אל'.,\tמעת |,\tממעטת | מ ממע'.,,1\tמפקידה | מ פקיד'.,\tלפקידה | מ לפקיד'.,\tומפקידה | מ ומפקי'.,\tלפקידה | ר לפקיד' מ לפקי'.,None
1,מקבילות,מ הרב'.,קו פא וחכ' ר במ2 וחכמים ק ד וחכמי' מ וחכמ'.,ק אומרי' ר אומרין במ2 מ או'.,,,,מ אל'.,במ2 מע'.,מ ממעט' בו2 ממעט.,,מ ד מפקיד'.,מ ד לפקיד'.,מ ומפקי' ד ומפקיד'.,מ לפקיד'.


In [160]:
def shift_col(col,beg):
    for i in range(beg,len(col)-1):
        col[i] = col[i+1]
    col = col[:-1]
    while col[-1].text == '':
        col = col[:-1]
    print("NEW LEN",len(col))
    return col
def shift_cols(pages,sample=False):
    output_pages = []
    for ip in range(len(pages)):
        page = pages[ip]
        max_len = max(len(page[1]),len(page[2]))
        min_len = min(len(page[1]),len(page[2]))
        print(ip,max_len,min_len)
        print("B",[p.text for p in page[2]])
        lastBblank= 0
        print("A",page[1][0].text)
        for i in range(min_len):
            if i > len(page[1])-1 or i > len(page[2])-1:
                break
            if page[2][i].text == '' and page[1][i].text != '':
                lastBblank = i
            if page[1][i].text == '' and page[2][i].text != '':
                print("Empty A line",i,"Last B blank",lastBblank)
                page[2] = shift_col(page[2],lastBblank)
        # if len(page[2])> len(page[1]):
        #     print("Extra B lines",len(page[2])-len(page[1])," Last B blank",lastBblank)
        #     shift_col(page[2],lastBblank)
        #     page[2] = page[2][:len(page[1])]
        if sample:
            print(len(page[1]),len(page[2]))
            df = pd.DataFrame([[p.text for p in page[1]],[p.text for p in page[2]]])    
            return df
shift_cols(pages_sample,sample=True)

0 14 14
B ['מקבילות', "מ הרב'.", "קו פא וחכ' ר במ2 וחכמים ק ד וחכמי' מ וחכמ'.", "ק אומרי' ר אומרין במ2 מ או'.", '', '', "מ אל'.", "במ2 מע'.", "מ ממעט' בו2 ממעט.", '', "מ ד מפקיד'.", "מ ד לפקיד'.", "מ ומפקי' ד ומפקיד'.", "מ לפקיד'."]
A 	כ"י של עדיות
14 14


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,"\tכ""י של עדיות",29\tהרבה | מ הרב'.,\tוחכמ' | פ וחכמי' ר א ק ש יא מ ד וחכמים.,\tאומ' | פ מ או' ר ש יא אומרין ק אומרי' ד אומרים.,\tכדברי2 | מ כדב'.,,30\tאלא | מ אל'.,\tמעת |,\tממעטת | מ ממע'.,,1\tמפקידה | מ פקיד'.,\tלפקידה | מ לפקיד'.,\tומפקידה | מ ומפקי'.,\tלפקידה | ר לפקיד' מ לפקי'.
1,מקבילות,מ הרב'.,קו פא וחכ' ר במ2 וחכמים ק ד וחכמי' מ וחכמ'.,ק אומרי' ר אומרין במ2 מ או'.,,,מ אל'.,במ2 מע'.,מ ממעט' בו2 ממעט.,,מ ד מפקיד'.,מ ד לפקיד'.,מ ומפקי' ד ומפקיד'.,מ לפקיד'.


In [161]:
all_pages = process_document(full_file_path,verbose=False)
shift_cols(all_pages)
table_all2 = create_output_document(all_pages)
table_all2.save('/Users/ephraimmeiri/Downloads/R Wieder mishnah edition processing/Eduyot Critical Edition Output Table 2.docx')

Uneven lines 0 jump 1
Uneven lines 1 jump 0
Uneven lines 2 jump 0
Uneven lines 5 jump 0
Uneven lines 7 jump 0
Uneven lines 8 jump 1
Uneven lines 9 jump 0
Uneven lines 10 jump 0
Uneven lines 11 jump 2
Uneven lines 12 jump 1
Uneven lines 13 jump 0
Uneven lines 14 jump 1
Uneven lines 15 jump 4
Uneven lines 16 jump 3
Uneven lines 17 jump 0
Uneven lines 18 jump 0
Uneven lines 22 jump 1
Uneven lines 23 jump 0
Uneven lines 24 jump 1
Uneven lines 25 jump 0
Uneven lines 26 jump 2
Uneven lines 27 jump 0
Uneven lines 28 jump 0
Uneven lines 29 jump 1
Uneven lines 30 jump 0
Uneven lines 31 jump 1
Uneven lines 32 jump 0
Uneven lines 33 jump 0
Uneven lines 34 jump 0
Uneven lines 36 jump 0
Uneven lines 39 jump 0
Uneven lines 40 jump 2
Uneven lines 41 jump 0
Uneven lines 42 jump 0
Uneven lines 45 jump 2
Uneven lines 49 jump 0
Uneven lines 50 jump 1
Uneven lines 51 jump 0
Uneven lines 52 jump 0
Uneven lines 53 jump 1
Uneven lines 54 jump 0
Uneven lines 56 jump 0
Uneven lines 57 jump 1
Uneven lines 58 ju

In [78]:
for page in all_pages:
    match_sv(page)

SV not in body כדברי2
SV not in body משנה ב
SV not in body משנה ג
SV not in body משנה ד
SV not in body שאדם חייב לומר כלשון רבו
SV not in body תשעת קבין
SV not in body משנה ה
SV not in body משנה ו
SV not in body את דברי היחיד ... שאין בית דין (שור' 27)
SV not in body ובמיניין גM
SV not in body משנה ז
SV not in body פל' גM
SV not in body שמעתה גM
SV not in body משנה ח
SV not in body בין2
SV not in body מרוב הביניין או
SV not in body הביניין גM
SV not in body משנה ט
SV not in body הלל ... בטומאה
SV not in body משנה י
SV not in body כל גM
SV not in body שקל גM
SV not in body ושקל גM
SV not in body משנה יא
SV not in body ובדינר מעות  ... בשלשה דינרין כסף (שור' 26)
SV not in body משנה יב
SV not in body שלכסא גM
SV not in body העשוי בה
SV not in body משנה יג
SV not in body תינשא גM
SV not in body אחת הבאה ... ממדינת הים (שור' 12)
SV not in body הזיתין גM
SV not in body דיברו גת
SV not in body משנה יד
SV not in body תינשא ותיטול גת
SV not in body היתרתם גת
SV not in body החמורה
SV not in body

IndexError: list index out of range

In [ ]:
all_pages = process_document(full_file_path)

In [103]:
table_all = create_output_document(all_pages)
table_all.save('/Users/ephraimmeiri/Downloads/R Wieder mishnah edition processing/Eduyot Critical Edition Output Table.docx')

In [ ]:
def combine_cols(page):
    if len(page[1]) != len(page[2]):
        print("Uneven columns")
        return
    
    for i in range(len(page[1])):
        

In [204]:
def match_sv(page,sample=False):
    body = [p.text for p in page[0]]
    cola = [p.text for p in page[1][1:]] # remove the header
    # colb = page[2][1:] # remove the header
    body_txt= " ".join(body).replace("\t"," ")
    # remove any multiple spaces
    body_txt = ' '.join(body_txt.split())
    # remove digit characters
    body_txt = ''.join([i for i in body_txt if not i.isdigit()])
    body_index= 0
    remaining_body = body_txt
    body_matched=[]
    for i in range(len(cola)):
        if cola[i] == '':
            body_matched.append('')
            continue
        sv, varts = None, None
        splt = cola[i].split("|")
        if len(splt) == 1:
            sv = splt[0].strip()
        else:
            sv = splt[0].strip()
            varts = splt[1]
        if "\t" in sv:
            sv= sv.split("\t")[1]
        if "..." in sv:
            sv = sv.split("...")[0].strip()
        geniza= ['גש1', 'גש4', 'גש3', 'גש5', 'גA', 'גJ1', 'גM', 'ג3J', 'גש6', 'גJ2', 'גת']
        while len(sv.split())>1 and sv.split()[-1] in geniza:
            print(sv," Geniza??")   
            sv = " ".join(sv.split()[:-1]).strip()
        if sv and sv[-1].isdigit():
            sv = sv[:-1]
        if sv not in body_txt:
            print("SV not in body",sv)
            body_matched.append('?')
            continue
        if sv in remaining_body:
            body_index = remaining_body.index(sv)
            body_matched.append(remaining_body[:body_index+len(sv)])
            remaining_body = remaining_body[body_index+len(sv):]
        else:
            body_index= body_txt.rindex(sv)
            # print("Need to go back to index",body_index)
            body_matched.append("*")
    if sample:
        return pd.DataFrame([body_matched,cola])
pages_sample = process_document(sample_file_path)
shift_cols(pages_sample)
match_sv(pages_sample[0],sample=True)

Uneven lines 0 jump 0
4 14 15 4
Uneven lines 1 jump 0
3 29 30 4
Uneven lines 2 jump 0
3 32 33 2
0 15 14
B ['מקבילות', "מ הרב'.", "קו פא וחכ' ר במ2 וחכמים ק ד וחכמי' מ וחכמ'.", "ק אומרי' ר אומרין במ2 מ או'.", '', '', '', "מ אל'.", "במ2 מע'.", "מ ממעט' בו2 ממעט.", '', "מ ד מפקיד'.", "מ ד לפקיד'.", "מ ומפקי' ד ומפקיד'.", "מ לפקיד'."]
A 	כ"י של עדיות
Empty A line 9 Last B blank 6
NEW LEN 14
1 30 29
B ['מקבילות', 'בו2 ממעט.', '', 'קו ק וכל ד ולכל.', "מ אש'.", '', '', 'מ ד בו וסת.', 'ד בו דיה.', "מ שעת'.", "קו ק ד והמשמשת פא ר ירל פסיל·ו ירו בו פסבו והמשמשת מ והמשמ' [פסבמ|המשמש'].", '', '', "קו בעידים פא בעדין ר ק ד ירל·ו פסיל·ו בו פסבמ·ו בעדים מ בעדי'.", '', "פא ירל·ו הממעטת ק ד ממעטת מ ממעט'.", '', '', "מ מפקיד'.", '', '', '', 'תו שמיי תע במ בו שמאי.', "תו במ או'.", 'תו כקב.', '', 'תו בית הלל תע במ והילל בו והלל.', "תו·ע אומ'.", 'תו מקבים.', '']
A 	כת"י של עדיות
Empty A line 13 Last B blank 12
NEW LEN 28
2 33 32
B ['מקבילות', 'תו במ וחכמים.', '', "תו·ע אומ'.", '', '', '', '', 'תו חיב תע בו

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,מסכזתהז עזדיזותז פר' א' הרבה,וחכמ',אומ',לא כדברי,,זה ולא כדברי זה אלא,מעת,לעת ממעטת,,על יד מפקידה,לפקידה,ומפקידה,לפקידה
1,29\tהרבה | מ הרב'.,\tוחכמ' | פ וחכמי' ר א ק ש יא מ ד וחכמים.,\tאומ' | פ מ או' ר ש יא אומרין ק אומרי' ד אומרים.,\tכדברי2 | מ כדב'.,,30\tאלא | מ אל'.,\tמעת |,\tממעטת | מ ממע'.,,1\tמפקידה | מ פקיד'.,\tלפקידה | מ לפקיד'.,\tומפקידה | מ ומפקי'.,\tלפקידה | ר לפקיד' מ לפקי'.


In [205]:
for page in pages_sample:
    match_sv(page)

SV not in body משנה ב
SV not in body משנה ג
SV not in body משנה ד


In [206]:
for page in all_pages:
    match_sv(page)

SV not in body משנה ב
SV not in body משנה ג
SV not in body משנה ד
SV not in body שאדם חייב לומר כלשון רבו
SV not in body תשעת קבין
SV not in body משנה ה
SV not in body משנה ו
ובמיניין גM  Geniza??
SV not in body משנה ז
פל' גM  Geniza??
שמעתה גM  Geniza??
SV not in body משנה ח
SV not in body מרוב הביניין או
הביניין גM  Geniza??
SV not in body משנה ט
SV not in body משנה י
כל גM  Geniza??
שקל גM  Geniza??
ושקל גM  Geniza??
SV not in body משנה יא
SV not in body משנה יב
שלכסא גM  Geniza??
SV not in body העשוי בה
SV not in body משנה יג
תינשא גM  Geniza??
הזיתין גM  Geniza??
דיברו גת  Geniza??
SV not in body משנה יד
תינשא ותיטול גת  Geniza??
היתרתם גת  Geniza??
SV not in body החמורה
תיטלי גת  Geniza??
SV not in body משנה יה
תיקנתם גת  Geniza??
תיקנתם גת  Geniza??
SV not in body אינו יכול
SV not in body משנה יו
חרס גת  Geniza??
SV not in body אכלים ומשקים שבתוכו
SV not in body בית הלל
SV not in body משנה ב
שניטמא גת  Geniza??
SV not in body משנה ג
טריפה גת  Geniza??
הן גת  Geniza??
SV not in b

In [212]:
second_words= set()
for page in all_pages:
    cola= [p.text for p in page[1][1:]]
    for i in range(len(cola)):
        sv, varts = None, None
        splt = cola[i].split(" |")
        if len(splt) == 1:
            sv = splt[0]
        else:
            sv = splt[0]
            varts = splt[1]
        if "\t" in sv:
            sv= sv.split("\t")[1]
        if "..." in sv:
            sv = sv.split("...")[0].strip()
        if len(sv.split())>1:
            second_words.add(sv.split()[1])
            if "משנה" in sv:
                print(len(sv.split()),sv)
print([w for w in second_words if w[0]=='ג'])

2 משנה ב
2 משנה ג
2 משנה ד
2 משנה ה
2 משנה ו
2 משנה ז
2 משנה ח
2 משנה ט
2 משנה י
2 משנה יא
2 משנה יב
2 משנה יג
2 משנה יד
2 משנה יה
2 משנה יו
2 משנה ב
2 משנה ג
2 משנה ד
2 משנה ה
2 משנה ו
2 משנה ז
2 משנה ח
2 משנה ט
2 משנה י
2 משנה יא
2 משנה ב
2 משנה ג
2 משנה ד'
2 משנה ה
2 משנה ו
2 משנה ז
2 משנה ח
2 משנה ט
2 משנה י
2 משנה יא
2 משנה יב
2 משנה יג
2 משנה יד
2 משנה ד
2 משנה ה
2 משנה ו
2 משנה ט
2 משנה י
2 משנה יא
2 משנה ב
2 משנה ג
2 משנה ד'
2 משנה ה'
2 משנה ז
2 משנה ח
2 משנה ט
2 משנה ג
2 משנה ד
2 משנה ה
2 משנה ו
2 משנה ח
2 משנה ט
2 משנה י
2 משנה יא
2 משנה ב
2 משנה ז
2 משנה ח
2 משנה ט
['גילגילון', 'גוג', "גמליא'", 'גש1', 'גש4', 'גת|', 'גש3', 'גש5', 'ג', 'גA', 'גמליאל', 'גJ1', 'גM', 'ג3J', 'גש6', 'גJ2', 'גת']


In [213]:
from docx import Document
from docx.shared import Inches

def match_sv(page):
    body = [p.text for p in page[0]]
    cola = page[1][1:]  # remove the header
    colb = page[2][1:]  # remove the header
    body_txt = " ".join(body).replace("\t", " ")
    body_txt = ' '.join(body_txt.split())
    body_txt = ''.join([i for i in body_txt if not i.isdigit()])
    
    remaining_body = body_txt
    body_matched = []
    
    for i in range(len(cola)):
        if cola[i].text == '':
            body_matched.append('')
            continue
        
        sv, varts = None, None
        splt = cola[i].text.split("|")
        if len(splt) == 1:
            sv = splt[0].strip()
        else:
            sv = splt[0].strip()
            varts = splt[1].strip()
        if "..." in sv:
            sv = sv.split("...")[0].strip()
        geniza= ['גש1', 'גש4', 'גש3', 'גש5', 'גA', 'גJ1', 'גM', 'ג3J', 'גש6', 'גJ2', 'גת']
        while len(sv.split())>1 and sv.split()[-1] in geniza:
            print(sv," Geniza??")   
            sv = " ".join(sv.split()[:-1]).strip()
        if sv.split()[0].strip()=="משנה" and len(sv.split())==2:
            if sv.split()[1][-1]=="'":
                sv = sv.split()[1]
            else:
                sv = sv.split()[1]+"'"
        if sv and sv[-1].isdigit():
            sv = sv[:-1]        
        if "\t" in sv:
            sv = sv.split("\t")[1]
        if sv not in remaining_body:
            print("SV not in body", sv)
            body_matched.append('?')
        else:
            body_index = remaining_body.index(sv)
            body_matched.append(remaining_body[:body_index + len(sv)])
            remaining_body = remaining_body[body_index + len(sv):]
    
    return body_matched, cola, colb

def create_output_document(pges):
    output_doc = Document()
    
    for page_num, page in enumerate(pges, 1):
        output_doc.add_heading(f'Page {page_num}', level=1)
        
        body_matched, cola, colb = match_sv(page)
        
        # Create table for columns
        table = output_doc.add_table(rows=len(body_matched) + 1, cols=2)
        table.style = 'Table Grid'
        
        # Add headers
        headers = table.rows[0].cells
        headers[0].text = "Body Matched"
        headers[1].text = "Combined Columns"
        
        # Add content
        for i in range(len(body_matched)):
            row = table.rows[i + 1].cells
            row[0].text = body_matched[i]
            
            # Create combined column
            combined_para = row[1].add_paragraph()
            # Copy cola
            for run in cola[i].runs:
                new_run = combined_para.add_run(run.text)
                copy_run_format(new_run, run)
            # Add separator
            combined_para.add_run(" // ")
            # Copy colb if it exists
            if i < len(colb):
                for run in colb[i].runs:
                    new_run = combined_para.add_run(run.text)
                    copy_run_format(new_run, run)
        
        # Add parallels
        if page[3]:
            output_doc.add_paragraph("Parallels:")
            for para in page[3]:
                new_para = output_doc.add_paragraph()
                copy_paragraph_format(new_para, para)
                for run in para.runs:
                    new_run = new_para.add_run(run.text)
                    copy_run_format(new_run, run)
        
        # Add page break
        output_doc.add_page_break()
    
    return output_doc

# Use the functions
input_file = '/Users/ephraimmeiri/Downloads/R Wieder mishnah edition processing/Sample Eduyot Critical Edition.docx'
output_file = '/Users/ephraimmeiri/Downloads/R Wieder mishnah edition processing/Sample Eduyot Critical Edition_output2.docx'

pges = process_document(input_file)
output_doc = create_output_document(pges)
output_doc.save(output_file)

print(f"Processed document saved as {output_file}")

Uneven lines 0 jump 0
4 14 15 4
Uneven lines 1 jump 0
3 29 30 4
Uneven lines 2 jump 0
3 32 33 2
SV not in body משנה ג
SV not in body וחכמ'
SV not in body כדבר'
SV not in body אלא
SV not in body ומחצה
SV not in body חייבין
SV not in body בחלה
SV not in body ומשהיגדילו
SV not in body המידות
SV not in body אמרו
SV not in body חמשת
SV not in body רבעים
SV not in body רבעים
SV not in body חייבין
SV not in body חייבין
SV not in body יוסה
SV not in body אומ'
SV not in body חמשה
SV not in body פטורין
SV not in body חמשה
SV not in body חייבין
SV not in body ד'
SV not in body הלל
SV not in body או'
Processed document saved as /Users/ephraimmeiri/Downloads/R Wieder mishnah edition processing/Sample Eduyot Critical Edition_output2.docx


In [214]:
input_file = '/Users/ephraimmeiri/Downloads/R Wieder mishnah edition processing/Eduyot Critical Edition.docx'
output_file = '/Users/ephraimmeiri/Downloads/R Wieder mishnah edition processing/Eduyot Critical Edition_output2.docx'

pges = process_document(input_file)
output_doc = create_output_document(pges)
output_doc.save(output_file)

print(f"Processed document saved as {output_file}")

Uneven lines 0 jump 1
4 26 28 4
Uneven lines 1 jump 0
3 29 30 4
Uneven lines 2 jump 0
3 32 33 2
3 33 33 0
3 29 12 0
Uneven lines 5 jump 0
3 25 26 5
4 30 11 0
Uneven lines 7 jump 0
3 29 30 5
Uneven lines 8 jump 1
3 25 27 4
Uneven lines 9 jump 0
3 29 30 4
Uneven lines 10 jump 0
3 33 34 3
Uneven lines 11 jump 2
3 29 32 1
Uneven lines 12 jump 1
3 28 30 1
Uneven lines 13 jump 0
4 31 32 4
Uneven lines 14 jump 1
3 30 32 4
Uneven lines 15 jump 4
3 28 33 1
Uneven lines 16 jump 3
3 26 30 4
Uneven lines 17 jump 0
3 28 29 5
Uneven lines 18 jump 0
3 28 29 1
3 27 28 0
3 38 35 0
3 31 31 0
Uneven lines 22 jump 1
3 31 33 1
Uneven lines 23 jump 0
3 25 26 1
Uneven lines 24 jump 1
5 31 33 3
Uneven lines 25 jump 0
3 27 28 9
Uneven lines 26 jump 2
3 31 34 1
Uneven lines 27 jump 0
3 29 30 4
Uneven lines 28 jump 0
3 26 27 5
Uneven lines 29 jump 1
3 30 32 4
Uneven lines 30 jump 0
3 27 28 4
Uneven lines 31 jump 1
3 29 31 4
Uneven lines 32 jump 0
3 30 31 5
Uneven lines 33 jump 0
3 30 31 4
Uneven lines 34 jump 0


IndexError: list index out of range